In [1]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd

import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, Normalize
import matplotlib.patches as patches

%matplotlib inline
%config InlineBackend.figure_format='retina'

plt.style.use('physrev_noticks.mplstyle')
plt.rcParams.update({"figure.figsize": [3.25, 1.01]})
plt.rcParams['figure.dpi'] = "300"

model_names = ["SubATTM", "SubCTRW", "SubFBM", "SubSBM", "SupFBM", "SupLW", "SupSBM", "STDBM"]
stat_names = ["AC","CS","SG","VD"]


In [2]:
total_confusion_dict = {}
for stat_name in stat_names:
    total_confusion_dict[stat_name] = {}
    for model_name_1 in model_names:
        total_confusion_dict[stat_name][model_name_1] = {}
        for model_name_2 in model_names:

            if total_confusion_dict[stat_name][model_name_1].get(model_name_2) is None:
                total_confusion_dict[stat_name][model_name_1][model_name_2] = 0

            for save_tag in range(5):

                data_dict = dict(np.load(f"./analysis_results/feature_ablation/disturb_res_dict_mse_{save_tag}.npy", allow_pickle=True).item())
                base_dict = dict(np.load(f"./analysis_results/feature_ablation/disturb_res_dict_mse_base_{save_tag}.npy", allow_pickle=True).item())
                model_results = data_dict[model_name_1][stat_name]
                base_results = base_dict[model_name_1][stat_name]

                # Update the count value
                accuracy = (np.sum(model_results == np.ones_like(model_results) * model_names.index(model_name_2)) / len(model_results))
                base_accuracy = (np.sum(base_results == np.ones_like(base_results) * model_names.index(model_name_2)) / len(base_results))
                accuracy_drop = base_accuracy - accuracy
                total_confusion_dict[stat_name][model_name_1][model_name_2] += accuracy_drop
for stat_name in stat_names:
    for model_name_1 in model_names:
        for model_name_2 in model_names:
            total_confusion_dict[stat_name][model_name_1][model_name_2] /= 5
        sub_dict = total_confusion_dict[stat_name][model_name_1]
        values = list(sub_dict.values())
        sorted_indices  = np.argsort(values)
        sorted_keys_by_indices = [list(sub_dict.keys())[i] for i in sorted_indices]
        
        total_confusion_dict[stat_name][model_name_1]["Sorted"] = sorted_keys_by_indices

In [3]:
total_confusion_dict_inverse = {}
for stat_name in stat_names:
    total_confusion_dict_inverse[stat_name] = {}
    for model_name_1 in model_names:
        total_confusion_dict_inverse[stat_name][model_name_1] = {}
        for model_name_2 in model_names:

            if total_confusion_dict_inverse[stat_name][model_name_1].get(model_name_2) is None:
                total_confusion_dict_inverse[stat_name][model_name_1][model_name_2] = 0

            for save_tag in range(5):

                data_dict = dict(np.load(f"./analysis_results/feature_ablation/disturb_res_dict_mse_inverse_{save_tag}.npy", allow_pickle=True).item())
                base_dict = dict(np.load(f"./analysis_results/feature_ablation/disturb_res_dict_mse_inverse_base_{save_tag}.npy", allow_pickle=True).item())
                model_results = data_dict[model_name_1][stat_name]
                base_results = base_dict[model_name_1][stat_name]

                # Update the count value
                accuracy = (np.sum(model_results == np.ones_like(model_results) * model_names.index(model_name_2)) / len(model_results))
                base_accuracy = (np.sum(base_results == np.ones_like(base_results) * model_names.index(model_name_2)) / len(base_results))
                accuracy_drop = base_accuracy - accuracy
                total_confusion_dict_inverse[stat_name][model_name_1][model_name_2] += accuracy_drop

for stat_name in stat_names:
    for model_name_1 in model_names:
        for model_name_2 in model_names:
            total_confusion_dict_inverse[stat_name][model_name_1][model_name_2] /= 5
        sub_dict = total_confusion_dict_inverse[stat_name][model_name_1]
        values = list(sub_dict.values())
        sorted_indices  = np.argsort(values)
        sorted_keys_by_indices = [list(sub_dict.keys())[i] for i in sorted_indices]
        
        total_confusion_dict_inverse[stat_name][model_name_1]["Sorted"] = sorted_keys_by_indices

In [4]:
results_dict = {}
for stat_name in stat_names:
        results_dict[stat_name] = {}
        for model_name_1 in model_names:
            top1_model_name = total_confusion_dict[stat_name][model_name_1]["Sorted"][0]
            top2_model_name = total_confusion_dict[stat_name][model_name_1]["Sorted"][1]
            if top1_model_name == "STDBM":
                model_name = "BM"
            else:
                model_name = top1_model_name
            top1_str = f"{model_name}\n{-total_confusion_dict[stat_name][model_name_1][top1_model_name]*100:.2f}"
            top2_str = f"{top2_model_name} {-total_confusion_dict[stat_name][model_name_1][top2_model_name]*100:.2f}"
            
            results_dict[stat_name][model_name_1] = top1_str

np.save("analysis_results/feature_ablation/total_confusion_dict.npy", results_dict)


In [5]:
results_dict_inverse = {}
for stat_name in stat_names:
        results_dict_inverse[stat_name] = {}
        for model_name_1 in model_names:
            top1_model_name = total_confusion_dict_inverse[stat_name][model_name_1]["Sorted"][0]
            top2_model_name = total_confusion_dict_inverse[stat_name][model_name_1]["Sorted"][1]
            if top1_model_name == "STDBM":
                model_name = "BM"
            else:
                model_name = top1_model_name
            top1_str = f"{model_name}\n{-total_confusion_dict_inverse[stat_name][model_name_1][top1_model_name]*100:.2f}"
            top2_str = f"{top2_model_name} {-total_confusion_dict_inverse[stat_name][model_name_1][top2_model_name]*100:.2f}"
            
            results_dict_inverse[stat_name][model_name_1] = top1_str

np.save("analysis_results/feature_ablation/total_confusion_dict_inverse.npy", results_dict_inverse)



In [6]:
# Additional code to analyze; Usage in paper

In [7]:
total_results = []
for i in range(5):
    results = np.load(f"./analysis_results/feature_ablation/disturb_res_dict_mse_inverse_{i}.npy", allow_pickle=True).item()
    total_results.extend(results["SupLW"]["SG"])
total_results = np.array(total_results)

In [8]:
# get percentage of each class
unique, counts = np.unique(total_results, return_counts=True)
# except for label 5
counts = counts[unique != 5]
unique = unique[unique != 5]
den = len(total_results) - np.sum(total_results == 5)
percentage = counts / den * 100
percentage_dict = dict(zip(unique, percentage))
percentage_dict # get most misclassification results

{0: 10.706903679206285,
 1: 2.6043819760231504,
 4: 79.24762298470442,
 6: 7.069036792062835,
 7: 0.37205456800330716}

In [9]:
print("Misclassification percentages (excluding correct classifications):")
print(f"SupFBM (most): {percentage_dict[4]:.1f}")

Misclassification percentages (excluding correct classifications):
SupFBM (most): 79.2


In [10]:
# Large accuracy drop

In [11]:
total_confusion_dict_inverse["VD"]["SubATTM"]

{'SubATTM': 0.13496428058687676,
 'SubCTRW': -0.03377961085845009,
 'SubFBM': -0.0008026171009983081,
 'SubSBM': -0.00824045402701485,
 'SupFBM': -0.03805155299732581,
 'SupLW': -0.011769678567304955,
 'SupSBM': -0.005060691422491588,
 'STDBM': -0.03725967561329115,
 'Sorted': ['SupFBM',
  'STDBM',
  'SubCTRW',
  'SupLW',
  'SubSBM',
  'SupSBM',
  'SubFBM',
  'SubATTM']}

In [12]:
# Acc drop percentage
# Sum over NOT subATTM drop

SupFBMratio = total_confusion_dict_inverse["VD"]["SubATTM"]["SupFBM"]/total_confusion_dict_inverse["VD"]["SubATTM"]["SubATTM"]
BMratio = total_confusion_dict_inverse["VD"]["SubATTM"]["STDBM"]/total_confusion_dict_inverse["VD"]["SubATTM"]["SubATTM"]
SubCTRWratio = total_confusion_dict_inverse["VD"]["SubATTM"]["SubCTRW"]/total_confusion_dict_inverse["VD"]["SubATTM"]["SubATTM"]

In [13]:
# print ratios with .1f and abs
print(f"SupFBM ratio: {abs(SupFBMratio*100):.1f}%")
print(f"BM ratio: {abs(BMratio*100):.1f}%")
print(f"SubCTRW ratio: {abs(SubCTRWratio*100):.1f}%")

SupFBM ratio: 28.2%
BM ratio: 27.6%
SubCTRW ratio: 25.0%
